In [2]:
# ============================================================
# E5a — Decision-Entropy Matched SFT (FIXED)
# L = L_SFT + λ * ( mean(H_student_dec) - mean(H_teacher_dec) )^2
# - teacher entropy masked to DECISION steps (via chosen token char spans)
# - student entropy masked to DECISION tokens (via offset_mapping)
# ============================================================

!pip -q install -U transformers peft accelerate bitsandbytes sentencepiece

import os, json, re
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from google.colab import drive

drive.mount("/content/drive")

# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = "/content/drive/MyDrive/DL_Local"
PROMPTS_PATH = os.path.join(BASE_DIR, "clinical_pharm_prompts_10000.jsonl")
TEACHER_PATH = os.path.join(BASE_DIR, "clinical_pharm_teacher_train_6000.jsonl")

OUT_DIR = os.path.join(BASE_DIR, "e5a_decision_entropy_sft")  # ✅ E5a dir
os.makedirs(OUT_DIR, exist_ok=True)

MAX_SEQ_LEN = 2048
EPOCHS = 2
LR = 2e-4
BATCH = 1
GRAD_ACC = 32
SEED = 42
LAMBDA = 0.05

STUDENTS = {
    "qwen25_1p5b_base":     {"path": "Qwen/Qwen2.5-1.5B",          "is_instruct": False},
    "qwen25_1p5b_instruct": {"path": "Qwen/Qwen2.5-1.5B-Instruct", "is_instruct": True},
    "qwen25_3b_base":       {"path": "Qwen/Qwen2.5-3B",            "is_instruct": False},
    "qwen25_3b_instruct":   {"path": "Qwen/Qwen2.5-3B-Instruct",   "is_instruct": True},
    "qwen25_7b_base":       {"path": "Qwen/Qwen2.5-7B",            "is_instruct": False},
    "qwen25_7b_instruct":   {"path": "Qwen/Qwen2.5-7B-Instruct",   "is_instruct": True},
}

# ----------------------------
# Helpers
# ----------------------------
def load_jsonl(p):
    with open(p, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def entropy_from_logprobs(logps_1d: torch.Tensor) -> torch.Tensor:
    probs = torch.softmax(logps_1d, dim=0)
    return -(probs * torch.log(probs + 1e-9)).sum()

DEC_PAT = [r"\bDecision\s*:\s*", r'"decision"\s*:\s*']

def find_decision_span_chars(teacher_text: str):
    for p in DEC_PAT:
        m = re.search(p, teacher_text)
        if m:
            return (m.end(), len(teacher_text))
    return None

def step_char_spans_from_chosen_tokens(logprobs_steps):
    """
    Build per-step [start,end) char spans by concatenating chosen token strings.
    This lets us mask teacher steps that overlap the Decision section span.
    """
    spans = []
    buf = ""
    for st in logprobs_steps or []:
        tok = (st.get("chosen") or {}).get("token", "")
        start = len(buf)
        buf += tok
        end = len(buf)
        spans.append((start, end))
    return spans

def teacher_section_entropy_mean(row, section_span):
    """
    Mean teacher entropy over steps that overlap section_span in chars.
    If no overlap or no logprobs, returns 0.
    """
    steps = row.get("logprobs_steps", []) or []
    if not steps or not section_span:
        return torch.tensor(0.0)

    spans = step_char_spans_from_chosen_tokens(steps)
    s0, s1 = section_span

    ent = []
    for i, st in enumerate(steps):
        a, b = spans[i]
        # overlap?
        if not (b <= s0 or a >= s1):
            topk = st.get("topk", [])
            if topk:
                lp = torch.tensor([t["logprob"] for t in topk], dtype=torch.float32)
                ent.append(entropy_from_logprobs(lp))

    return torch.stack(ent).mean() if ent else torch.tensor(0.0)

# ----------------------------
# Load data
# ----------------------------
prompts = {r["id"]: r["prompt"] for r in load_jsonl(PROMPTS_PATH)}
rows = [r for r in load_jsonl(TEACHER_PATH) if r.get("status") == "ok" and r.get("teacher_text")]

print(f"Loaded train teacher rows: {len(rows)}")

# ----------------------------
# Dataset
# ----------------------------
class E5aDataset(Dataset):
    def __init__(self, rows, tok, is_instr):
        self.rows, self.tok, self.is_instr = rows, tok, is_instr

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        prompt = prompts[r["id"]]
        ans = r["teacher_text"]

        if self.is_instr and hasattr(self.tok, "apply_chat_template"):
            ptxt = self.tok.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            ptxt = prompt

        full = ptxt + ans
        enc = self.tok(
            full,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_offsets_mapping=True,
            add_special_tokens=True
        )

        ids = enc["input_ids"]
        offs = enc["offset_mapping"]
        attn = enc.get("attention_mask", [1]*len(ids))

        # SFT labels (all answer tokens)
        labels = [-100] * len(ids)
        answer_start = len(ptxt)
        for i, (s, e) in enumerate(offs):
            if s >= answer_start:
                labels[i] = ids[i]

        # Teacher mean entropy on DECISION section
        dec_span = find_decision_span_chars(ans)
        ent_teacher = teacher_section_entropy_mean(r, dec_span)

        # Build DECISION token mask for STUDENT (char-based on ans, mapped into full offsets)
        dec_mask = torch.zeros(len(ids), dtype=torch.bool)
        if dec_span:
            ds, de = dec_span
            # decision span in full-text char coords = answer_start + [ds,de)
            full_ds = answer_start + ds
            full_de = answer_start + de
            for i, (s, e) in enumerate(offs):
                # token overlaps section?
                if labels[i] != -100 and not (e <= full_ds or s >= full_de):
                    dec_mask[i] = True

        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "dec_mask": dec_mask,                 # ✅ student decision positions
            "ent_teacher": ent_teacher.float(),   # ✅ scalar
        }

# ----------------------------
# Collator (pads + stacks)
# ----------------------------
class PadCollator:
    def __init__(self, pad_id: int):
        self.pad_id = pad_id

    def __call__(self, features):
        max_len = max(f["input_ids"].shape[0] for f in features)

        def pad_1d(x, pad_val):
            if x.shape[0] == max_len:
                return x
            return torch.cat([x, torch.full((max_len - x.shape[0],), pad_val, dtype=x.dtype)], dim=0)

        input_ids = torch.stack([pad_1d(f["input_ids"], self.pad_id) for f in features], dim=0)
        attention_mask = torch.stack([pad_1d(f["attention_mask"], 0) for f in features], dim=0)
        labels = torch.stack([pad_1d(f["labels"], -100) for f in features], dim=0)

        dec_mask = torch.stack([pad_1d(f["dec_mask"].to(torch.long), 0).bool() for f in features], dim=0)
        ent_teacher = torch.stack([f["ent_teacher"] for f in features], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "dec_mask": dec_mask,
            "ent_teacher": ent_teacher,
        }

# ----------------------------
# Trainer
# ----------------------------
class E5aTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        dec_mask = inputs.pop("dec_mask")
        ent_teacher = inputs.pop("ent_teacher")  # [B]

        out = model(**inputs)
        logits = out.logits  # [B,T,V]

        # CE on answer tokens only (labels!=-100), shifted
        sl = logits[:, :-1, :]
        ll = labels[:, 1:]
        ce = torch.nn.CrossEntropyLoss(ignore_index=-100)(
            sl.reshape(-1, sl.size(-1)), ll.reshape(-1)
        )

        # Student entropy per token (shifted to match sl time)
        probs = torch.softmax(sl, dim=-1)
        ent_tok = -(probs * torch.log(probs + 1e-9)).sum(-1)  # [B,T-1]

        # Align dec_mask to ent_tok time (drop first position)
        dec_mask_t = dec_mask[:, 1:]  # [B,T-1]
        # Mean entropy over decision tokens only (fallback to 0 if none)
        ent_student = torch.zeros(ent_teacher.shape[0], device=logits.device)
        for b in range(ent_teacher.shape[0]):
            m = dec_mask_t[b]
            if m.any():
                ent_student[b] = ent_tok[b][m].mean()

        # Entropy match penalty
        penalty = ((ent_student - ent_teacher.to(logits.device)) ** 2).mean()

        loss = ce + LAMBDA * penalty
        return (loss, out) if return_outputs else loss

# ----------------------------
# Run
# ----------------------------
for name, cfg in STUDENTS.items():
    print(f"\n===== E5a TRAINING {name} =====")
    out_path = os.path.join(OUT_DIR, name)
    os.makedirs(out_path, exist_ok=True)

    tok = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg["path"],
        load_in_4bit=True,
        device_map="auto",
        trust_remote_code=True
    )
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj"],
        lora_dropout=0.05,
        task_type="CAUSAL_LM"
    ))
    model.train()

    ds = E5aDataset(rows, tok, cfg["is_instruct"])

    trainer = E5aTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=out_path,
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH,
            gradient_accumulation_steps=GRAD_ACC,
            learning_rate=LR,
            fp16=True,
            seed=SEED,
            save_strategy="no",
            remove_unused_columns=False,
            report_to="none",
        ),
        train_dataset=ds,
        data_collator=PadCollator(tok.pad_token_id),
    )

    trainer.train()
    model.save_pretrained(out_path)
    tok.save_pretrained(out_path)
    print(f"✅ Saved: {out_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded train teacher rows: 5000

===== E5a TRAINING qwen25_1p5b_base =====


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Step,Training Loss


✅ Saved: /content/drive/MyDrive/DL_Local/e5a_decision_entropy_sft/qwen25_1p5b_base

===== E5a TRAINING qwen25_1p5b_instruct =====


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Step,Training Loss


✅ Saved: /content/drive/MyDrive/DL_Local/e5a_decision_entropy_sft/qwen25_7b_instruct


In [4]:
# ============================================================
# E5b — Explanation-Entropy Matched SFT (FIXED)
# L = L_SFT + λ * ( mean(H_student_expl) - mean(H_teacher_expl) )^2
# ============================================================

import os, re, torch
from torch.utils.data import Dataset

OUT_DIR_E5B = os.path.join(BASE_DIR, "e5b_explanation_entropy_sft")  # ✅ separate folder
os.makedirs(OUT_DIR_E5B, exist_ok=True)

EXP_PAT = [r"\bExplanation\s*:\s*", r'"explanation"\s*:\s*']
class PadCollator:
    def __init__(self, tok, mask_key: str):
        self.tok = tok
        self.mask_key = mask_key

    def __call__(self, features):
        # pad input_ids / attention_mask
        batch = self.tok.pad(
            {
                "input_ids": [f["input_ids"] for f in features],
                "attention_mask": [f["attention_mask"] for f in features],
            },
            padding=True,
            return_tensors="pt"
        )
        max_len = batch["input_ids"].shape[1]

        def pad_1d(t, pad_value, dtype=None):
            if dtype is None:
                dtype = t.dtype
            out = torch.full((max_len,), pad_value, dtype=dtype)
            L = t.shape[0]
            out[:L] = t.to(dtype)
            return out

        # pad labels
        batch["labels"] = torch.stack(
            [pad_1d(f["labels"], -100, dtype=torch.long) for f in features],
            dim=0
        )

        # pad the requested mask (expl_mask here)
        batch[self.mask_key] = torch.stack(
            [pad_1d(f[self.mask_key].to(torch.long), 0, dtype=torch.long).bool() for f in features],
            dim=0
        )

        # ent_teacher: [B]
        batch["ent_teacher"] = torch.stack([f["ent_teacher"] for f in features], dim=0)

        return batch

def find_expl_span_chars(teacher_text: str):
    for p in EXP_PAT:
        m = re.search(p, teacher_text)
        if m:
            return (m.end(), len(teacher_text))
    return None

def teacher_expl_entropy_mean(row, section_span):
    # identical to E5a teacher_section_entropy_mean, just different span
    steps = row.get("logprobs_steps", []) or []
    if not steps or not section_span:
        return torch.tensor(0.0)

    spans = step_char_spans_from_chosen_tokens(steps)
    s0, s1 = section_span

    ent = []
    for i, st in enumerate(steps):
        a, b = spans[i]
        if not (b <= s0 or a >= s1):
            topk = st.get("topk", [])
            if topk:
                lp = torch.tensor([t["logprob"] for t in topk], dtype=torch.float32)
                ent.append(entropy_from_logprobs(lp))

    return torch.stack(ent).mean() if ent else torch.tensor(0.0)

class E5bDataset(Dataset):
    def __init__(self, rows, tok, is_instr):
        self.rows, self.tok, self.is_instr = rows, tok, is_instr

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        prompt = prompts[r["id"]]
        ans = r["teacher_text"]

        if self.is_instr and hasattr(self.tok, "apply_chat_template"):
            ptxt = self.tok.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            ptxt = prompt

        full = ptxt + ans
        enc = self.tok(
            full,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_offsets_mapping=True,
            add_special_tokens=True
        )

        ids = enc["input_ids"]
        offs = enc["offset_mapping"]
        attn = enc.get("attention_mask", [1]*len(ids))

        labels = [-100] * len(ids)
        answer_start = len(ptxt)
        for i, (s, e) in enumerate(offs):
            if s >= answer_start:
                labels[i] = ids[i]

        expl_span = find_expl_span_chars(ans)
        ent_teacher = teacher_expl_entropy_mean(r, expl_span)

        expl_mask = torch.zeros(len(ids), dtype=torch.bool)
        if expl_span:
            es, ee = expl_span
            full_es = answer_start + es
            full_ee = answer_start + ee
            for i, (s, e) in enumerate(offs):
                if labels[i] != -100 and not (e <= full_es or s >= full_ee):
                    expl_mask[i] = True

        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "expl_mask": expl_mask,
            "ent_teacher": ent_teacher.float(),
        }

class E5bTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        expl_mask = inputs.pop("expl_mask")
        ent_teacher = inputs.pop("ent_teacher")  # [B]

        out = model(**inputs)
        logits = out.logits

        sl = logits[:, :-1, :]
        ll = labels[:, 1:]
        ce = torch.nn.CrossEntropyLoss(ignore_index=-100)(
            sl.reshape(-1, sl.size(-1)), ll.reshape(-1)
        )

        probs = torch.softmax(sl, dim=-1)
        ent_tok = -(probs * torch.log(probs + 1e-9)).sum(-1)  # [B,T-1]

        expl_mask_t = expl_mask[:, 1:]
        ent_student = torch.zeros(ent_teacher.shape[0], device=logits.device)
        for b in range(ent_teacher.shape[0]):
            m = expl_mask_t[b]
            if m.any():
                ent_student[b] = ent_tok[b][m].mean()

        penalty = ((ent_student - ent_teacher.to(logits.device)) ** 2).mean()
        loss = ce + LAMBDA * penalty
        return (loss, out) if return_outputs else loss

# ----------------------------
# Run
# ----------------------------
for name, cfg in STUDENTS.items():
    print(f"\n===== E5b TRAINING {name} =====")
    out_path = os.path.join(OUT_DIR_E5B, name)
    os.makedirs(out_path, exist_ok=True)

    tok = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg["path"],
        load_in_4bit=True,
        device_map="auto",
        trust_remote_code=True
    )
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj"],
        lora_dropout=0.05,
        task_type="CAUSAL_LM"
    ))
    model.train()

    ds = E5bDataset(rows, tok, cfg["is_instruct"])

    trainer = E5bTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=out_path,
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH,
            gradient_accumulation_steps=GRAD_ACC,
            learning_rate=LR,
            fp16=True,
            seed=SEED,
            save_strategy="no",
            remove_unused_columns=False,
            report_to="none",
        ),
        train_dataset=ds,
        data_collator=PadCollator(tok, mask_key="expl_mask"),
    )

    trainer.train()
    model.save_pretrained(out_path)
    tok.save_pretrained(out_path)
    print(f"✅ Saved: {out_path}")



===== E5b TRAINING qwen25_1p5b_base =====


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss


✅ Saved: /content/drive/MyDrive/DL_Local/e5b_explanation_entropy_sft/qwen25_1p5b_base

===== E5b TRAINING qwen25_1p5b_instruct =====


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


...

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss


✅ Saved: /content/drive/MyDrive/DL_Local/e5b_explanation_entropy_sft/qwen25_7b_instruct


In [5]:
# ============================================================
# COLAB — Unassign runtime (official API)
# ============================================================

from google.colab import runtime

print("✅ Job finished. Unassigning runtime...")
runtime.unassign()


✅ Job finished. Unassigning runtime...
